# 🤖 AI/ML Wikipedia RAG Chatbot

**Internship Project — Phase 2: Build the RAG Pipeline**

This notebook builds a complete Retrieval-Augmented Generation (RAG) system over 12 Wikipedia articles covering core AI/ML concepts. It:
1. Fetches the source articles directly from Wikipedia
2. Chunks and embeds them with Google Gemini, storing vectors in ChromaDB
3. Builds a LangGraph pipeline that retrieves relevant chunks and generates grounded answers using Groq (Llama 3.3 70B)
4. Evaluates the system on 20 test questions using a RAGAS-style scoring approach (faithfulness, answer relevancy, context precision), judged by Groq

**Before running:** add your API keys as Colab Secrets (click the 🔑 key icon in the left sidebar):
- `GEMINI_KEY` — your Google AI Studio / Gemini API key
- `GROQ_KEY` — your Groq API key

Then run every cell from top to bottom. No code edits are required — every cell already ran successfully end-to-end.

## 1. Install Dependencies

Installs every library needed for fetching Wikipedia content, chunking, Gemini embeddings, ChromaDB, the Groq client, LangGraph orchestration, and RAGAS-related evaluation tooling.

In [ ]:
# CELL 1 — Install all required libraries
!pip install langchain langchain-community langchain-google-genai chromadb pypdf groq langgraph ragas datasets wikipedia-api tenacity -q

## 2. Load API Keys

Reads `GEMINI_KEY` and `GROQ_KEY` from Colab Secrets and sets them as environment variables. **Keys are never hardcoded or printed.** Wrapped in error handling so a missing secret produces a clear message instead of a raw traceback.

In [ ]:
# CELL 2 — Imports and API key setup
try:
    import os
    import time
    import warnings
    warnings.filterwarnings("ignore")

    from google.colab import userdata

    # Load API keys from Colab Secrets
    os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_KEY")
    os.environ["GROQ_API_KEY"]   = userdata.get("GROQ_KEY")

    print("✅ API keys loaded successfully")

except ImportError as e:
    print(f"❌ Import Error: {e}")

except KeyError as e:
    print(f"❌ Missing Key Error: {e}")

except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")

## 3. Fetch the Document Corpus (12 Wikipedia Articles)

Downloads 12 Wikipedia articles covering foundational AI/ML topics — RAG itself, LLMs, transformers, embeddings, CNNs, RNNs, attention, BERT, GPT-4, knowledge graphs, vector databases, and RLHF. Each article is capped at 8000 characters and saved as a `.txt` file so the corpus stays a manageable size for chunking and embedding.

In [ ]:
# CELL 3 — Fetch Wikipedia articles as document corpus (10-20 documents)

try:
    import wikipediaapi
    import os

    wiki = wikipediaapi.Wikipedia(
        language="en",
        user_agent="RAGInternshipBot/1.0 (internship project)"
    )

    # 12 AI/ML Wikipedia articles — your document set
    TOPICS = [
        "Retrieval-augmented generation",
        "Large language model",
        "Transformer (deep learning architecture)",
        "Word embedding",
        "Convolutional neural network",
        "Recurrent neural network",
        "Attention (machine learning)",
        "BERT (language model)",
        "GPT-4",
        "Knowledge graph",
        "Vector database",
        "Reinforcement learning from human feedback",
    ]

    os.makedirs("/content/docs", exist_ok=True)

    fetched = []
    for topic in TOPICS:
        try:
            page = wiki.page(topic)

            if page.exists():
                safe_name = topic.replace("/", "_").replace(" ", "_")
                filepath = f"/content/docs/{safe_name}.txt"

                with open(filepath, "w", encoding="utf-8") as f:
                    # Write first 8000 chars to keep chunks manageable
                    f.write(page.text[:8000])

                fetched.append(filepath)
                print(f"✅ Saved: {topic}")

            else:
                print(f"⚠️ Not found: {topic}")

        except Exception as e:
            print(f"❌ Error processing '{topic}': {e}")

    print(f"\n📚 Total documents fetched: {len(fetched)}")

except ImportError as e:
    print(f"❌ Import Error: {e}")

except PermissionError as e:
    print(f"❌ Permission Error: {e}")

except OSError as e:
    print(f"❌ File System Error: {e}")

except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")

## 4. Load and Chunk the Documents

Loads every saved `.txt` file, tags each with its source article name as metadata, and splits the text into overlapping 500-character chunks (50-character overlap) using `RecursiveCharacterTextSplitter`. Smaller chunks keep retrieval focused and reduce the token cost of each embedding call.

In [ ]:
# CELL 4 — Ingestion: Load → Chunk → Embed (Gemini) → Store (ChromaDB)

try:
    import os
    import time
    from langchain_community.document_loaders import TextLoader
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    from langchain_google_genai import GoogleGenerativeAIEmbeddings
    from langchain_community.vectorstores import Chroma
    from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type
    import google.api_core.exceptions

    # ── Step 1: Load all documents ──────────────────────────────────────────────
    all_docs = []
    doc_dir = "/content/docs"

    for filename in os.listdir(doc_dir):
        if filename.endswith(".txt"):
            try:
                filepath = os.path.join(doc_dir, filename)
                loader = TextLoader(filepath, encoding="utf-8")
                docs = loader.load()

                # Attach source metadata
                for doc in docs:
                    doc.metadata["source"] = filename.replace(".txt", "").replace("_", " ")

                all_docs.extend(docs)
                print(f"📄 Loaded: {filename}  ({len(docs[0].page_content)} chars)")

            except Exception as e:
                print(f"❌ Error loading '{filename}': {e}")

    print(f"\n✅ Total documents loaded: {len(all_docs)}")

    # ── Step 2: Split into ~500-word chunks ────────────────────────────────────
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
        separators=["\n\n", "\n", ".", " ", ""]
    )

    chunks = splitter.split_documents(all_docs)
    print(f"✅ Total chunks created: {len(chunks)}")

    # Preview first chunk
    try:
        print(f"\n🔍 Sample chunk:\n{chunks[0].page_content[:300]}...")
        print(f"   Metadata: {chunks[0].metadata}")
    except IndexError:
        print("⚠️ No chunks available to preview.")

except ImportError as e:
    print(f"❌ Import Error: {e}")

except FileNotFoundError as e:
    print(f"❌ File Not Found Error: {e}")

except PermissionError as e:
    print(f"❌ Permission Error: {e}")

except OSError as e:
    print(f"❌ Operating System Error: {e}")

except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")

## 5. Embed Chunks and Store in ChromaDB

Embeds every chunk using Gemini's `gemini-embedding-001` model and stores the vectors in a native ChromaDB collection called `rag_documents`.

**Important implementation detail:** this uses the raw `google-generativeai` SDK (`genai.embed_content`) instead of LangChain's `GoogleGenerativeAIEmbeddings` wrapper, because that wrapper called an outdated `v1beta` endpoint that returned a 404 for this model. Calling the SDK directly avoids that bug entirely.

Chunks are embedded in batches of 20 with automatic retry and exponential backoff if the Gemini free-tier quota is hit (`429 RESOURCE_EXHAUSTED`).

In [ ]:
# CELL 5 — Embed with google-generativeai SDK directly (using models/gemini-embedding-001)

try:
    import os
    import time
    import google.generativeai as genai
    import chromadb
    import builtins

    genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

    CHROMA_DIR = "/content/chroma_db"
    EMBEDDING_MODEL = "models/gemini-embedding-001"   # ← confirmed from diagnostic

    # ── Custom embedding class using SDK directly (bypasses langchain v1beta bug) ─
    class GeminiEmbeddings:
        def __init__(self, model_name: str):
            self.model_name = model_name

        def embed_documents(self, texts: list) -> list:
            vectors = []
            for text in texts:
                try:
                    result = genai.embed_content(
                        model=self.model_name,
                        content=text,
                        task_type="retrieval_document"
                    )
                    vectors.append(result["embedding"])

                except Exception as e:
                    print(f"❌ Error embedding document: {e}")
                    raise

            return vectors

        def embed_query(self, text: str) -> list:
            try:
                result = genai.embed_content(
                    model=self.model_name,
                    content=text,
                    task_type="retrieval_query"
                )
                return result["embedding"]

            except Exception as e:
                print(f"❌ Error embedding query: {e}")
                raise

    embeddings = GeminiEmbeddings(EMBEDDING_MODEL)

    # ── Quick sanity test ─────────────────────────────────────────────────────────
    try:
        print("🔍 Testing embedding model …")
        test_vec = embeddings.embed_query("hello world")
        print(f"✅ Embedding works! Vector dimension: {len(test_vec)}")

    except Exception as e:
        print(f"❌ Embedding model test failed: {e}")
        raise

    # ── Batch embed + store in ChromaDB ──────────────────────────────────────────
    BATCH_SIZE = 20

    def embed_in_batches(chunks, embeddings_obj, persist_dir, batch_size=20):
        try:
            client = chromadb.PersistentClient(path=persist_dir)

            # Clear existing collection if re-running cell
            try:
                client.delete_collection("rag_documents")
                print("  🗑️  Cleared existing collection")
            except Exception:
                pass

            collection = client.create_collection(
                name="rag_documents",
                metadata={"hnsw:space": "cosine"}
            )

            total = len(chunks)
            doc_id = 0

            for i in range(0, total, batch_size):
                batch = chunks[i: i + batch_size]

                print(
                    f"  Batch {i//batch_size + 1}/{(total + batch_size - 1)//batch_size} "
                    f"(chunks {i+1}–{min(i+batch_size, total)} of {total})",
                    end=" ... "
                )

                for attempt in range(5):
                    try:
                        texts = [c.page_content for c in batch]
                        metadatas = [c.metadata for c in batch]
                        ids = [f"doc_{doc_id + j}" for j in range(len(batch))]
                        vectors = embeddings_obj.embed_documents(texts)

                        collection.add(
                            documents=texts,
                            embeddings=vectors,
                            metadatas=metadatas,
                            ids=ids
                        )

                        doc_id += len(batch)
                        print("✅")
                        break

                    except Exception as e:
                        err_str = str(e)

                        if (
                            "429" in err_str
                            or "RESOURCE_EXHAUSTED" in err_str
                            or "quota" in err_str.lower()
                        ):
                            wait = 60 * (attempt + 1)
                            print(
                                f"\n  ⚠️  Quota hit — waiting {wait}s (retry {attempt+1}/5)",
                                end=" "
                            )
                            time.sleep(wait)

                        else:
                            print(f"\n  ❌ Unexpected error: {e}")
                            raise

                time.sleep(1)   # polite pause between batches

            return collection, client

        except Exception as e:
            print(f"❌ Error during batch embedding/storage: {e}")
            raise

    print(f"\n🚀 Starting ingestion — {len(chunks)} chunks into ChromaDB …\n")

    collection, chroma_client = embed_in_batches(
        chunks,
        embeddings,
        CHROMA_DIR,
        BATCH_SIZE
    )

    print(f"\n✅ ChromaDB built at     : {CHROMA_DIR}")
    print(f"✅ Total chunks stored   : {collection.count()}")

    # ── Save to builtins so later cells can reuse without re-embedding ────────────
    try:
        builtins.gemini_embeddings = embeddings
        builtins.chroma_collection = collection
        builtins.chroma_client_obj = chroma_client
        builtins.embedding_model_name = EMBEDDING_MODEL

        print(f"\n✅ Saved to builtins — ready for Cell 6, 7, 8")

    except Exception as e:
        print(f"❌ Error saving objects to builtins: {e}")
        raise

except ImportError as e:
    print(f"❌ Import Error: {e}")

except KeyError as e:
    print(f"❌ Missing Environment Variable: {e}")

except FileNotFoundError as e:
    print(f"❌ File Not Found Error: {e}")

except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")

## 6. Query Pipeline — Retrieve + Generate (Sanity Test)

Defines the core RAG functions: `retrieve_chunks` (embed the question, search ChromaDB for the top-5 most similar chunks), `generate_answer` (build a grounded prompt and call Groq's `llama-3.3-70b-versatile`), and `ask_question` (the full pipeline). Ends with a live test question to confirm everything works before building the LangGraph version.

In [ ]:
# CELL 6 — Query: retrieve top-5 chunks + generate answer with Groq

try:
    import chromadb
    from groq import Groq
    import os
    import builtins

    CHROMA_DIR = "/content/chroma_db"

    # ── Load collection + embeddings ──────────────────────────────────────────────
    chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
    collection = chroma_client.get_collection("rag_documents")
    embeddings = builtins.gemini_embeddings

    groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

    def retrieve_chunks(question: str, k: int = 5) -> dict:
        """Embed the question and retrieve top-k chunks from ChromaDB."""
        try:
            query_vec = embeddings.embed_query(question)

            results = collection.query(
                query_embeddings=[query_vec],
                n_results=k,
                include=["documents", "metadatas", "distances"]
            )

            return results

        except Exception as e:
            print(f"❌ Error retrieving chunks: {e}")
            raise

    def generate_answer(question: str, documents: list, metadatas: list) -> str:
        """Build prompt from retrieved chunks and call Groq."""
        try:
            context = "\n\n---\n\n".join(
                [
                    f"[Source: {m.get('source', 'Unknown')}]\n{doc}"
                    for doc, m in zip(documents, metadatas)
                ]
            )

            prompt = f"""You are a helpful AI assistant. Answer the question using ONLY the context provided below.
Cite the source for every fact you state. If the context does not contain enough information, say so clearly.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

            response = groq_client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.2,
                max_tokens=512
            )

            return response.choices[0].message.content

        except Exception as e:
            print(f"❌ Error generating answer: {e}")
            raise

    def ask_question(question: str, k: int = 5) -> dict:
        """Full RAG pipeline: embed → retrieve → generate."""
        try:
            results = retrieve_chunks(question, k=k)

            documents = results["documents"][0]
            metadatas = results["metadatas"][0]

            answer = generate_answer(question, documents, metadatas)

            sources = [m.get("source", "Unknown") for m in metadatas]

            return {
                "question": question,
                "answer": answer,
                "chunks": documents,
                "sources": sources
            }

        except Exception as e:
            print(f"❌ Error in RAG pipeline: {e}")
            raise

    # ── Test with a sample question ───────────────────────────────────────────────
    try:
        print("🚀 Testing RAG query pipeline …\n")

        result = ask_question("What is retrieval-augmented generation?")

        print(f"❓ Question : {result['question']}")
        print(f"\n💬 Answer  :\n{result['answer']}")
        print(f"\n📎 Sources  : {list(set(result['sources']))}")
        print(f"\n📄 Chunks   : {len(result['chunks'])} retrieved")

    except Exception as e:
        print(f"❌ Test query failed: {e}")

except ImportError as e:
    print(f"❌ Import Error: {e}")

except KeyError as e:
    print(f"❌ Missing Environment Variable: {e}")

except AttributeError as e:
    print(f"❌ Builtins Object Error: {e}")

except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")

## 7. LangGraph Pipeline — Connecting Ingestion and Query Into One Flow

Wraps the retrieve/generate logic from Section 6 into a LangGraph `StateGraph` with three nodes: `retrieve`, `generate`, and `error_handler`. A conditional edge routes to `error_handler` if retrieval fails, instead of crashing or returning a blank answer. This is the production version of the pipeline used by the Streamlit app in Phase 3.

In [ ]:
# CELL 7 — LangGraph pipeline (retrieve → generate → error_handler)

try:
    from langgraph.graph import StateGraph, END
    from typing import TypedDict, List, Optional
    from groq import Groq
    import chromadb
    import os
    import builtins

    # ── State schema ──────────────────────────────────────────────────────────────
    class RAGState(TypedDict):
        question: str
        chunks: List[str]
        sources: List[str]
        answer: str
        error: Optional[str]
        vectorstore: Optional[object]

    # ── Shared resources ──────────────────────────────────────────────────────────
    _embeddings = builtins.gemini_embeddings
    _chroma_client = chromadb.PersistentClient(path="/content/chroma_db")
    _collection = _chroma_client.get_collection("rag_documents")
    _groq = Groq(api_key=os.environ["GROQ_API_KEY"])

    # ── Node 1: Retrieve ──────────────────────────────────────────────────────────
    def retrieve_node(state: RAGState) -> RAGState:
        """Retrieve top-5 relevant chunks from ChromaDB."""
        try:
            query_vec = _embeddings.embed_query(state["question"])

            results = _collection.query(
                query_embeddings=[query_vec],
                n_results=5,
                include=["documents", "metadatas"]
            )

            docs = results["documents"][0]
            metas = results["metadatas"][0]

            return {
                **state,
                "chunks": docs,
                "sources": [m.get("source", "Unknown") for m in metas],
                "error": None
            }

        except Exception as e:
            return {
                **state,
                "chunks": [],
                "sources": [],
                "error": f"Retrieval error: {e}"
            }

    # ── Node 2: Generate ──────────────────────────────────────────────────────────
    def generate_node(state: RAGState) -> RAGState:
        """Generate answer using Groq based on retrieved chunks."""

        if state.get("error"):
            return state

        if not state["chunks"]:
            return {
                **state,
                "answer": "No relevant chunks found.",
                "error": "No chunks retrieved."
            }

        context = "\n\n---\n\n".join(
            [
                f"[Source: {src}]\n{chunk}"
                for src, chunk in zip(state["sources"], state["chunks"])
            ]
        )

        prompt = f"""You are a helpful AI assistant. Answer the question using ONLY the context provided below.
Cite the source for every fact you state. If the context does not contain enough information, say so clearly.

CONTEXT:
{context}

QUESTION: {state['question']}

ANSWER:"""

        try:
            response = _groq.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.2,
                max_tokens=512
            )

            return {
                **state,
                "answer": response.choices[0].message.content,
                "error": None
            }

        except Exception as e:
            return {
                **state,
                "answer": "",
                "error": f"Generation error: {e}"
            }

    # ── Node 3: Error handler ─────────────────────────────────────────────────────
    def error_handler_node(state: RAGState) -> RAGState:
        """Handle errors gracefully and return a user-friendly message."""
        try:
            return {
                **state,
                "answer": f"⚠️ Pipeline error: {state.get('error', 'Unknown error')}. "
                          "Please check your API keys and document store."
            }

        except Exception as e:
            return {
                **state,
                "answer": f"⚠️ Error handler failed: {e}"
            }

    # ── Router: decide whether to generate or handle error ───────────────────────
    def route_after_retrieval(state: RAGState) -> str:
        try:
            return "error_handler" if state.get("error") else "generate"

        except Exception:
            return "error_handler"

    # ── Build the LangGraph ───────────────────────────────────────────────────────
    try:
        builder = StateGraph(RAGState)

        builder.add_node("retrieve", retrieve_node)
        builder.add_node("generate", generate_node)
        builder.add_node("error_handler", error_handler_node)

        builder.set_entry_point("retrieve")

        builder.add_conditional_edges(
            "retrieve",
            route_after_retrieval,
            {
                "generate": "generate",
                "error_handler": "error_handler"
            }
        )

        builder.add_edge("generate", END)
        builder.add_edge("error_handler", END)

        rag_pipeline = builder.compile()

        print("✅ LangGraph RAG pipeline compiled successfully")
        print("   Nodes : retrieve → (generate | error_handler) → END\n")

    except Exception as e:
        print(f"❌ Pipeline compilation failed: {e}")
        raise

    # ── Run a test question through the pipeline ──────────────────────────────────
    try:
        print("=" * 60)
        print("🚀 Running LangGraph pipeline …")
        print("=" * 60 + "\n")

        initial_state: RAGState = {
            "question": "What is a large language model?",
            "chunks": [],
            "sources": [],
            "answer": "",
            "error": None,
            "vectorstore": None
        }

        output = rag_pipeline.invoke(initial_state)

        print(f"❓ Question : {output['question']}")
        print(f"\n💬 Answer  :\n{output['answer']}")
        print(f"\n📎 Sources  : {list(set(output['sources']))}")
        print(f"\n📄 Chunks   : {len(output['chunks'])} retrieved")

        if output.get("error"):
            print(f"\n⚠️  Error   : {output['error']}")
        else:
            print("\n✅ Pipeline ran successfully with no errors")

    except Exception as e:
        print(f"❌ Pipeline execution failed: {e}")

except ImportError as e:
    print(f"❌ Import Error: {e}")

except AttributeError as e:
    print(f"❌ Missing builtins object: {e}")

except KeyError as e:
    print(f"❌ Missing environment variable: {e}")

except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")

## 8. RAGAS-Style Evaluation (20 Test Questions)

Runs 20 hand-written test questions through the RAG pipeline and scores each answer on three RAGAS-style metrics — **faithfulness**, **answer relevancy**, and **context precision** — using Groq itself as the judge LLM.

**Why not the official `ragas` library directly?** The installed version of `ragas` pulled in a broken dependency on a deprecated LangChain VertexAI module, which raised import errors that could not be resolved cleanly in this Colab environment. Using Groq as a structured judge produces the same three metrics with the same scoring rubric, without that dependency conflict.

In [ ]:
# CELL 8 — RAGAS-style evaluation using Groq

try:
    import pandas as pd
    import time
    from groq import Groq
    import os

    groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

    # ── 20 test questions ─────────────────────────────────────────────────────────
    TEST_QUESTIONS = [
        "What is retrieval-augmented generation?",
        "What is a large language model?",
        "How do transformers work in deep learning?",
        "What is word embedding in NLP?",
        "Explain convolutional neural networks.",
        "What is a recurrent neural network?",
        "How does the attention mechanism work in machine learning?",
        "What is BERT and what is it used for?",
        "What are the key features of GPT-4?",
        "What is a knowledge graph?",
        "What is a vector database?",
        "What is reinforcement learning from human feedback?",
        "How does chunking help in RAG systems?",
        "What role do embeddings play in retrieval systems?",
        "What is the difference between semantic search and keyword search?",
        "How does RAG reduce hallucinations in AI systems?",
        "What are the main components of a transformer architecture?",
        "How is context precision measured in RAG evaluation?",
        "What is the purpose of a vector store in a RAG pipeline?",
        "How does self-attention differ from traditional attention?",
    ]

    # ── Helper: score one answer with Groq ───────────────────────────────────────
    def score_answer(question: str, answer: str, context: str) -> dict:
        """
        Ask Groq to score faithfulness and answer relevancy on a 0-1 scale.
        Returns dict with faithfulness, answer_relevancy, context_precision.
        """

        scoring_prompt = f"""You are an expert RAG evaluator. Score the following RAG output strictly.

QUESTION: {question}

RETRIEVED CONTEXT:
{context[:1500]}

GENERATED ANSWER:
{answer[:800]}

Score each metric from 0.0 to 1.0 (two decimal places):
1. faithfulness       — Is the answer fully supported by the context? (1.0 = fully grounded, 0.0 = hallucinated)
2. answer_relevancy   — Does the answer directly address the question? (1.0 = perfectly relevant, 0.0 = off-topic)
3. context_precision  — Does the retrieved context contain information needed to answer? (1.0 = highly relevant context, 0.0 = irrelevant)

Reply in this EXACT format and nothing else:
faithfulness: 
answer_relevancy: 
context_precision: """

        for attempt in range(3):
            try:
                response = groq_client.chat.completions.create(
                    model="llama-3.3-70b-versatile",
                    messages=[{"role": "user", "content": scoring_prompt}],
                    temperature=0.0,
                    max_tokens=100
                )

                raw = response.choices[0].message.content.strip()

                # Parse the three scores
                scores = {}

                for line in raw.splitlines():
                    if "faithfulness:" in line:
                        scores["faithfulness"] = float(line.split(":")[1].strip())

                    elif "answer_relevancy:" in line:
                        scores["answer_relevancy"] = float(line.split(":")[1].strip())

                    elif "context_precision:" in line:
                        scores["context_precision"] = float(line.split(":")[1].strip())

                if len(scores) == 3:
                    return scores

            except Exception as e:
                if attempt < 2:
                    time.sleep(10)
                else:
                    print(f"❌ Scoring failed after 3 attempts: {e}")

        # Fallback if parsing fails
        return {
            "faithfulness": 0.0,
            "answer_relevancy": 0.0,
            "context_precision": 0.0
        }

    # ── Run pipeline + scoring for all 20 questions ───────────────────────────────
    print("🔄 Running RAG pipeline + scoring for 20 questions …\n")

    records = []

    for i, question in enumerate(TEST_QUESTIONS, 1):
        print(
            f"  [{i:02d}/20] {question[:65]}{'...' if len(question) > 65 else ''}",
            end=" "
        )

        try:
            # Step 1: RAG answer
            rag_result = ask_question(question, k=5)
            answer = rag_result["answer"]
            chunks = rag_result["chunks"]
            sources = rag_result["sources"]
            context = "\n\n".join(chunks)

            # Step 2: Score with Groq
            scores = score_answer(question, answer, context)

            records.append({
                "question": question,
                "answer": answer[:120] + "..." if len(answer) > 120 else answer,
                "sources": ", ".join(set(sources)),
                "faithfulness": scores["faithfulness"],
                "answer_relevancy": scores["answer_relevancy"],
                "context_precision": scores["context_precision"],
            })

            print(
                f"✅  F={scores['faithfulness']:.2f}  "
                f"AR={scores['answer_relevancy']:.2f}  "
                f"CP={scores['context_precision']:.2f}"
            )

        except Exception as e:
            print(f"❌ Error: {e}")

            records.append({
                "question": question,
                "answer": "Error",
                "sources": "",
                "faithfulness": 0.0,
                "answer_relevancy": 0.0,
                "context_precision": 0.0,
            })

        time.sleep(1)   # avoid Groq rate limit

    # ── Build results DataFrame ───────────────────────────────────────────────────
    try:
        df = pd.DataFrame(records)
        df.index = df.index + 1   # start index at 1

        print("\n" + "=" * 90)
        print("📊 RAGAS-STYLE EVALUATION RESULTS — 20 TEST QUESTIONS")
        print("=" * 90)

        print(
            df[
                [
                    "question",
                    "faithfulness",
                    "answer_relevancy",
                    "context_precision"
                ]
            ].to_string()
        )

        print("\n" + "=" * 90)
        print("📈 AGGREGATE SCORES")
        print("=" * 90)

        print(f"  Faithfulness        : {df['faithfulness'].mean():.3f}")
        print(f"  Answer Relevancy    : {df['answer_relevancy'].mean():.3f}")
        print(f"  Context Precision   : {df['context_precision'].mean():.3f}")

        print(
            f"  Overall Average     : "
            f"{df[['faithfulness','answer_relevancy','context_precision']].mean().mean():.3f}"
        )

    except Exception as e:
        print(f"❌ Error creating evaluation DataFrame: {e}")
        raise

    # Save for Cell 9
    try:
        import builtins

        builtins.eval_df = df
        print("\n✅ Results saved — proceed to Cell 9")

    except Exception as e:
        print(f"❌ Error saving results to builtins: {e}")

except ImportError as e:
    print(f"❌ Import Error: {e}")

except KeyError as e:
    print(f"❌ Missing Environment Variable: {e}")

except NameError as e:
    print(f"❌ Required object not found: {e}")

except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")

## 9. Display the Final Evaluation Table

Displays the full 20-question scorecard plus aggregate averages for each metric, and highlights the best and worst performing questions by faithfulness score. On this run, the system achieved **0.975 faithfulness**, **0.980 answer relevancy**, and **0.818 context precision** — an overall average of **0.924**.

In [ ]:
# CELL 9 — Display final evaluation table

import builtins
import pandas as pd

df = builtins.eval_df

print("=" * 90)
print("📊 FULL EVALUATION TABLE")
print("=" * 90)
print(df.to_string())

print("\n" + "=" * 90)
print("📈 AGGREGATE SCORES")
print("=" * 90)
print(f"  Faithfulness        : {df['faithfulness'].mean():.3f}")
print(f"  Answer Relevancy    : {df['answer_relevancy'].mean():.3f}")
print(f"  Context Precision   : {df['context_precision'].mean():.3f}")
print(f"  Overall Average     : {df[['faithfulness','answer_relevancy','context_precision']].mean().mean():.3f}")

# Highlight best and worst
best  = df.loc[df['faithfulness'].idxmax(), 'question']
worst = df.loc[df['faithfulness'].idxmin(), 'question']
print(f"\n  🏆 Best  faithfulness : {best[:70]}")
print(f"  ⚠️  Worst faithfulness : {worst[:70]}")

## ✅ Notebook Complete

You have now built and evaluated a full RAG pipeline: Wikipedia ingestion → chunking → Gemini embeddings → ChromaDB → LangGraph (retrieve + generate via Groq) → RAGAS-style evaluation.

**Next step (Phase 3):** `rag_pipeline.py` in this repo wraps the LangGraph pipeline built in Section 7 into a reusable `ask_question()` function, used by the Streamlit app (`app.py`) for the public-facing chatbot. `ingest.py` reproduces Sections 3–5 as a standalone script for building `chroma_db/` outside of Colab.